<a href="https://colab.research.google.com/github/nithin12342/phase2/blob/main/ml_pipeline/h5_omnifusion/notebooks/H5_OmniFusion_Phase5_Precision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎯 H5-OmniFusion: Phase 5 Precision Optimization
**Target: F1 0.78+ | Precision 0.68+ | Recall 0.90+**

### ✨ What's New in Phase 5:
| Change | Old | New | Impact |
|--------|-----|-----|--------|
| Focal Alpha | 0.85 | **0.65** | ↑ Precision |
| Mixup Alpha | 0.2 | **0.4** | ↑ Generalization |
| Audio Dropout | 0.0 | **0.35** | ↓ Audio Noise |
| Ensemble | Simple Avg | **Weighted** | ↑ F1 Score |

---

## 🛠️ Step 1: Environment Setup (Run First)

In [1]:
import os, sys, shutil
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# 2. Configuration
REPO_URL = "https://github.com/nithin12342/phase2.git"
DRIVE_BASE = "/content/drive/MyDrive/DAIC-WOZ_Datasets"
CACHE_ZIP = f"{DRIVE_BASE}/mamba_env_v1.zip"
LOCAL_PKGS = "/content/local_pkgs"

# 3. Clone / Sync Project Code
%cd /content/
if os.path.exists('/content/phase2'):
    !rm -rf /content/phase2
!git clone {REPO_URL} /content/phase2

# 4. Setup Persistent Environment
os.makedirs(LOCAL_PKGS, exist_ok=True)
if os.path.exists(CACHE_ZIP):
    print("📦 Found cached environment on Drive! Unzipping...")
    !unzip -q {CACHE_ZIP} -d {LOCAL_PKGS}
    print("✅ Environment restored in seconds.")
else:
    print("⏳ No cache found. Building Mamba-SSM (~12-15 mins)...")
    !apt-get install -y ninja-build
    !pip install ninja packaging
    !MAX_JOBS=4 pip install --target={LOCAL_PKGS} mamba-ssm causal-conv1d>=1.4.0 --no-build-isolation
    !pip install --target={LOCAL_PKGS} transformers opensmile librosa

    print("💾 Backing up to Drive...")
    %cd {LOCAL_PKGS}
    !zip -r {CACHE_ZIP} .
    %cd /content/
    print(f"✅ Cache saved to {CACHE_ZIP}")

# 5. Link Paths
sys.path.insert(0, LOCAL_PKGS)
os.environ['PYTHONPATH'] = f"{LOCAL_PKGS}:/content/phase2/ml_pipeline/h5_omnifusion"
print("\n🚀 READY! Environment Setup Complete.")

Mounted at /content/drive
/content
Cloning into '/content/phase2'...
fatal: could not read Username for 'https://github.com': No such device or address
📦 Found cached environment on Drive! Unzipping...
✅ Environment restored in seconds.

🚀 READY! Environment Setup Complete.


## 🏋️ Step 2: 5-Fold Phase 5 Training (~30-40 min)
Training with optimized hyperparameters for maximum precision.

In [2]:
import os

# Paths
OS_PATH = "/content/phase2/ml_pipeline/h5_omnifusion"
ENV_PKGS = "/content/local_pkgs"
DATA_DIR = "/content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output"
LABELS = "/content/drive/MyDrive/DAIC-WOZ_Datasets/all_labels_perfect.csv"
SAVE_DIR = "/content/drive/MyDrive/DAIC-WOZ_Datasets/checkpoints_phase5"

!mkdir -p {SAVE_DIR}

# Phase 5 Training: All 5 folds
for fold in range(5):
    print(f"\n{'='*60}")
    print(f"🚀 PHASE 5 TRAINING: FOLD {fold}/4")
    print(f"{'='*60}\n")

    !PYTHONPATH={OS_PATH}:{ENV_PKGS} python {OS_PATH}/scripts/train.py \
        --data_dir {DATA_DIR} \
        --labels_csv {LABELS} \
        --tier medium \
        --fold {fold} \
        --epochs 30 \
        --lr 1e-4 \
        --output_dir {SAVE_DIR}

    print(f"\n✅ Fold {fold} Complete!")

print("\n🏁 ALL 5 FOLDS COMPLETE! Phase 5 Training Done.")


🚀 PHASE 5 TRAINING: FOLD 0/4

python3: can't open file '/content/phase2/ml_pipeline/h5_omnifusion/scripts/train.py': [Errno 2] No such file or directory

✅ Fold 0 Complete!

🚀 PHASE 5 TRAINING: FOLD 1/4

python3: can't open file '/content/phase2/ml_pipeline/h5_omnifusion/scripts/train.py': [Errno 2] No such file or directory

✅ Fold 1 Complete!

🚀 PHASE 5 TRAINING: FOLD 2/4

python3: can't open file '/content/phase2/ml_pipeline/h5_omnifusion/scripts/train.py': [Errno 2] No such file or directory

✅ Fold 2 Complete!

🚀 PHASE 5 TRAINING: FOLD 3/4

python3: can't open file '/content/phase2/ml_pipeline/h5_omnifusion/scripts/train.py': [Errno 2] No such file or directory

✅ Fold 3 Complete!

🚀 PHASE 5 TRAINING: FOLD 4/4

python3: can't open file '/content/phase2/ml_pipeline/h5_omnifusion/scripts/train.py': [Errno 2] No such file or directory

✅ Fold 4 Complete!

🏁 ALL 5 FOLDS COMPLETE! Phase 5 Training Done.


## 🗳️ Step 3: Weighted Ensemble Evaluation

In [3]:
import os

OS_PATH = "/content/phase2/ml_pipeline/h5_omnifusion"
ENV_PKGS = "/content/local_pkgs"
CHECKPOINTS = "/content/drive/MyDrive/DAIC-WOZ_Datasets/checkpoints_phase5"
DATA_DIR = "/content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output"

# Sync latest code
%cd /content/phase2/
!git pull origin main
%cd /content/

# Run Ensemble Prediction
print("🗳️ Running Phase 5 Ensemble Prediction...")
!PYTHONPATH={OS_PATH}:{ENV_PKGS} python {OS_PATH}/scripts/ensemble_predict.py \
    --checkpoints {CHECKPOINTS} \
    --input {DATA_DIR} \
    --tier medium \
    --output "/content/phase5_ensemble_results.csv"

# Final Evaluation with Weighted Voting
print("\n🏆 PHASE 5 FINAL RESULTS (Weighted Ensemble)...")
!PYTHONPATH={OS_PATH}:{ENV_PKGS} python {OS_PATH}/scripts/evaluate_ensemble.py

[Errno 2] No such file or directory: '/content/phase2/'
/content
fatal: not a git repository (or any of the parent directories): .git
/content
🗳️ Running Phase 5 Ensemble Prediction...
python3: can't open file '/content/phase2/ml_pipeline/h5_omnifusion/scripts/ensemble_predict.py': [Errno 2] No such file or directory

🏆 PHASE 5 FINAL RESULTS (Weighted Ensemble)...
python3: can't open file '/content/phase2/ml_pipeline/h5_omnifusion/scripts/evaluate_ensemble.py': [Errno 2] No such file or directory


## 📊 Step 4: Compare Phase 4 vs Phase 5 (Optional)

In [4]:
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

# Load results
try:
    df = pd.read_csv("/content/phase5_ensemble_results.csv")

    # Calculate metrics at different thresholds
    print("\n📊 THRESHOLD OPTIMIZATION RESULTS")
    print("="*50)

    for thresh in [0.45, 0.50, 0.52, 0.55, 0.58, 0.60]:
        preds = (df['prob'] >= thresh).astype(int)
        f1 = f1_score(df['label'], preds)
        prec = precision_score(df['label'], preds)
        rec = recall_score(df['label'], preds)

        marker = "⭐" if f1 > 0.75 else ""
        print(f"Threshold {thresh:.2f}: F1={f1:.4f} | Prec={prec:.4f} | Rec={rec:.4f} {marker}")

except Exception as e:
    print(f"⚠️ Could not load results: {e}")
    print("Run Step 3 first to generate ensemble results.")

⚠️ Could not load results: [Errno 2] No such file or directory: '/content/phase5_ensemble_results.csv'
Run Step 3 first to generate ensemble results.


## 🔬 Step 5: Ablation Study (Optional)

In [5]:
OS_PATH = "/content/phase2/ml_pipeline/h5_omnifusion"
ENV_PKGS = "/content/local_pkgs"

print("🔬 Running Modality Ablation Study...")
!PYTHONPATH={OS_PATH}:{ENV_PKGS} python {OS_PATH}/scripts/run_ablation_study.py

🔬 Running Modality Ablation Study...
python3: can't open file '/content/phase2/ml_pipeline/h5_omnifusion/scripts/run_ablation_study.py': [Errno 2] No such file or directory


---
## 🎯 Expected Results

| Metric | Phase 4 | Phase 5 Target |
|--------|---------|----------------|
| **F1 Score** | 0.72 | **0.78-0.82** |
| **Precision** | 0.58 | **0.68-0.72** |
| **Recall** | 0.95 | **0.90-0.95** |
| **False Positives** | 60 | **<35** |

In [7]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content
!rm -rf phase2
!git clone https://github.com/nithin12342/phase2.git
%cd /content/phase2 && git pull origin main
%cd /content

!pip install torch h5py pandas numpy scikit-learn tqdm --quiet


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content
Cloning into 'phase2'...
remote: Enumerating objects: 1922, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 1922 (delta 29), reused 73 (delta 22), pack-reused 1831 (from 2)
Receiving objects: 100% (1922/1922), 19.56 MiB | 16.84 MiB/s, done.
Resolving deltas: 100% (1011/1011), done.
[Errno 2] No such file or directory: '/content/phase2 && git pull origin main'
/content
/content


In [3]:
# Cell 1: Mount & Clone
from google.colab import drive
drive.mount('/content/drive')

%cd /content
!rm -rf phase2
!git clone https://github.com/nithin12342/phase2.git

# Cell 2: Pull Latest (separate cell)
%cd /content/phase2
!git pull origin main
%cd /content

# Cell 3: Install Dependencies
!pip install torch h5py pandas numpy scikit-learn tqdm --quiet
print("✅ Setup Complete!")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content
Cloning into 'phase2'...
remote: Enumerating objects: 1938, done.
remote: Counting objects: 100% (107/107), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 1938 (delta 41), reused 94 (delta 31), pack-reused 1831 (from 2)
Receiving objects: 100% (1938/1938), 19.72 MiB | 6.98 MiB/s, done.
Resolving deltas: 100% (1023/1023), done.
/content/phase2
From https://github.com/nithin12342/phase2
 * branch            main       -> FETCH_HEAD
Already up to date.
/content
✅ Setup Complete!


In [9]:
# Cell 4: Run Ensemble Evaluation
OS_PATH = "/content/phase2/ml_pipeline/h5_omnifusion"
CHECKPOINTS = "/content/drive/MyDrive/DAIC-WOZ_Datasets/checkpoints_phase4"
DATA_DIR = "/content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output"

# Run ensemble
!PYTHONPATH={OS_PATH} python {OS_PATH}/scripts/ensemble_predict.py \
    --checkpoints {CHECKPOINTS} \
    --input {DATA_DIR} \
    --tier medium \
    --output "/content/final_ensemble_results.csv"

# Evaluate (with new weighted ensemble)
!PYTHONPATH={OS_PATH} python {OS_PATH}/scripts/evaluate_ensemble.py


Utils loaded. Device: cuda
Available: librosa=True, opensmile=False, cv2=True, transformers=True
ModelLoader initialized. Pretrained path: /content/drive/MyDrive/DAIC-WOZ_Datasets/pretrained_models
Using device: cuda
✅ Model loaded from /content/drive/MyDrive/DAIC-WOZ_Datasets/checkpoints_phase4/h5_omnifusion_medium_fold0_best.pt
✅ Model loaded from /content/drive/MyDrive/DAIC-WOZ_Datasets/checkpoints_phase4/h5_omnifusion_medium_fold1_best.pt
✅ Model loaded from /content/drive/MyDrive/DAIC-WOZ_Datasets/checkpoints_phase4/h5_omnifusion_medium_fold2_best.pt
✅ Model loaded from /content/drive/MyDrive/DAIC-WOZ_Datasets/checkpoints_phase4/h5_omnifusion_medium_fold3_best.pt
✅ Model loaded from /content/drive/MyDrive/DAIC-WOZ_Datasets/checkpoints_phase4/h5_omnifusion_medium_fold4_best.pt
🚀 Loaded 5 checkpoints for ensemble.
Found 358 H5 files. Processing with ensemble...
✅ Saved ensemble results for 358 files to /content/final_ensemble_results.csv
🏆 Calculating Final Project Results...
💡 Deri

In [10]:
OUT_DIR = "/content/drive/MyDrive/DAIC-WOZ_Datasets/checkpoints_phase5"
LABELS = "/content/drive/MyDrive/DAIC-WOZ_Datasets/all_labels_perfect.csv"

for fold in range(5):
    !PYTHONPATH={OS_PATH} python {OS_PATH}/scripts/train.py \
        --data_dir {DATA_DIR} --labels_csv {LABELS} \
        --output_dir {OUT_DIR} --tier medium --epochs 30 --fold_idx {fold}


Utils loaded. Device: cuda
Available: librosa=True, opensmile=False, cv2=True, transformers=True
ModelLoader initialized. Pretrained path: /content/drive/MyDrive/DAIC-WOZ_Datasets/pretrained_models
Using device: cuda

Configuration:
  Tier: medium
  Dimension: 256
  Params: Audio=facebook/wav2vec2-large-xlsr-53, Text=mental/mental-roberta-base
  Folds: 5 (Current: 0)
  Labels CSV: /content/drive/MyDrive/DAIC-WOZ_Datasets/all_labels_perfect.csv

Initializing model...
  Total parameters: 11.98M

Loading data from /content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output...
  Using labels from /content/drive/MyDrive/DAIC-WOZ_Datasets/all_labels_perfect.csv
[H5Dataset] Loaded 358 labels from /content/drive/MyDrive/DAIC-WOZ_Datasets/all_labels_perfect.csv
[H5Dataset] Found 358 H5 files across subdirectories of /content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output
[H5Dataset] Discovered subdirectories: Extended-DAIC, DAIC-WOZ, EATD-Corpus
[H5Dataset] Loaded 358 external labels fr

In [11]:
# Cell: Evaluate Phase 5 Model
OS_PATH = "/content/phase2/ml_pipeline/h5_omnifusion"
PHASE5_CHECKPOINTS = "/content/drive/MyDrive/DAIC-WOZ_Datasets/checkpoints_phase5"
DATA_DIR = "/content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output"

# Run ensemble with NEW Phase 5 checkpoints
!PYTHONPATH={OS_PATH} python {OS_PATH}/scripts/ensemble_predict.py \
    --checkpoints {PHASE5_CHECKPOINTS} \
    --input {DATA_DIR} \
    --tier medium \
    --output "/content/phase5_ensemble_results.csv"

# Final evaluation
!PYTHONPATH={OS_PATH} python {OS_PATH}/scripts/evaluate_ensemble.py


Utils loaded. Device: cuda
Available: librosa=True, opensmile=False, cv2=True, transformers=True
ModelLoader initialized. Pretrained path: /content/drive/MyDrive/DAIC-WOZ_Datasets/pretrained_models
Using device: cuda
✅ Model loaded from /content/drive/MyDrive/DAIC-WOZ_Datasets/checkpoints_phase5/h5_omnifusion_medium_fold0_best.pt
✅ Model loaded from /content/drive/MyDrive/DAIC-WOZ_Datasets/checkpoints_phase5/h5_omnifusion_medium_fold1_best.pt
✅ Model loaded from /content/drive/MyDrive/DAIC-WOZ_Datasets/checkpoints_phase5/h5_omnifusion_medium_fold2_best.pt
✅ Model loaded from /content/drive/MyDrive/DAIC-WOZ_Datasets/checkpoints_phase5/h5_omnifusion_medium_fold3_best.pt
✅ Model loaded from /content/drive/MyDrive/DAIC-WOZ_Datasets/checkpoints_phase5/h5_omnifusion_medium_fold4_best.pt
🚀 Loaded 5 checkpoints for ensemble.
Found 358 H5 files. Processing with ensemble...
✅ Saved ensemble results for 358 files to /content/phase5_ensemble_results.csv
🏆 Calculating Final Project Results...
💡 Der

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Cell 1: Pull Phase 6 fixes
%cd /content/phase2
!git pull origin main
%cd /content

# Cell 2: Train with 15 epochs (prevents late degradation)
OS_PATH = "/content/phase2/ml_pipeline/h5_omnifusion"
DATA_DIR = "/content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output"
LABELS = "/content/drive/MyDrive/DAIC-WOZ_Datasets/all_labels_perfect.csv"
OUT_DIR = "/content/drive/MyDrive/DAIC-WOZ_Datasets/checkpoints_phase6"

for fold in range(5):
    print(f"\n{'='*50}")
    print(f"🚀 PHASE 6 - FOLD {fold}")
    print(f"{'='*50}\n")
    !PYTHONPATH={OS_PATH} python {OS_PATH}/scripts/train.py \
        --data_dir {DATA_DIR} \
        --labels_csv {LABELS} \
        --output_dir {OUT_DIR} \
        --tier medium \
        --epochs 15 \
        --fold_idx {fold}

# Cell 3: Evaluate Phase 6
!PYTHONPATH={OS_PATH} python {OS_PATH}/scripts/ensemble_predict.py \
    --checkpoints {OUT_DIR} --input {DATA_DIR} --tier medium \
    --output "/content/phase6_results.csv"

!PYTHONPATH={OS_PATH} python {OS_PATH}/scripts/evaluate_ensemble.py


/content/phase2
From https://github.com/nithin12342/phase2
 * branch            main       -> FETCH_HEAD
Already up to date.
/content

🚀 PHASE 6 - FOLD 0

Utils loaded. Device: cuda
Available: librosa=True, opensmile=False, cv2=True, transformers=True
ModelLoader initialized. Pretrained path: /content/drive/MyDrive/DAIC-WOZ_Datasets/pretrained_models
Using device: cuda

Configuration:
  Tier: medium
  Dimension: 256
  Params: Audio=facebook/wav2vec2-large-xlsr-53, Text=mental/mental-roberta-base
  Folds: 5 (Current: 0)
  Labels CSV: /content/drive/MyDrive/DAIC-WOZ_Datasets/all_labels_perfect.csv

Initializing model...
  Total parameters: 11.98M

Loading data from /content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output...
  Using labels from /content/drive/MyDrive/DAIC-WOZ_Datasets/all_labels_perfect.csv
[H5Dataset] Loaded 358 labels from /content/drive/MyDrive/DAIC-WOZ_Datasets/all_labels_perfect.csv
[H5Dataset] Found 358 H5 files across subdirectories of /content/drive/MyDrive/D